In [ ]:
# Sentiment Analysis using LSTM + Pre-trained Embeddings (TensorFlow/Keras)

In [ ]:
Architecture Overview 

Text → Tokenization → Padding → Embedding (GloVe/Word2Vec)
     → LSTM → Dropout → Dense → Output

# Step 1: Imports

In [1]:
import numpy as np
import pandas as pd

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout

from sklearn.model_selection import train_test_split


# Step 2: Simple Dataset (Editable)

In [3]:
# 1) Start small for teaching, later scale

# texts = [
#     "I love this movie",
#     "This film was terrible",
#     "Amazing experience",
#     "Worst acting ever",
#     "I enjoyed the film",
#     "Not good at all",
#     "Absolutely fantastic",
#     "I hate this",
#     "Very boring movie",
#     "Best film ever"
# ]

# labels = [1,0,1,0,1,0,1,0,0,1]


## Or use following later

In [ ]:
# 2) Use this later

df_train = pd.read_csv("temp/data_imdb_train.csv")
df_test = pd.read_csv("temp/data_imdb_test.csv")

texts = df_train["review"].values
labels = df_train["sentiment"].values

In [21]:
texts[:4]

array(["? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't

In [22]:
labels[:4]

array([1, 0, 0, 1], dtype=int64)

# Step 3: Tokenization

In [23]:
# took 20 sec.

vocab_size = 5000

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
word_index = tokenizer.word_index

print("Word Index:", word_index)

Word Index: {'<OOV>': 1, 'the': 2, 'and': 3, 'a': 4, 'of': 5, 'to': 6, 'is': 7, 'br': 8, 'in': 9, 'it': 10, 'i': 11, 'this': 12, 'that': 13, 'was': 14, 'as': 15, 'for': 16, 'with': 17, 'movie': 18, 'but': 19, 'film': 20, 'on': 21, 'not': 22, 'you': 23, 'are': 24, 'his': 25, 'have': 26, 'he': 27, 'be': 28, 'one': 29, 'all': 30, 'at': 31, 'by': 32, 'an': 33, 'they': 34, 'who': 35, 'so': 36, 'from': 37, 'like': 38, 'her': 39, 'or': 40, 'just': 41, 'about': 42, "it's": 43, 'out': 44, 'if': 45, 'has': 46, 'some': 47, 'there': 48, 'what': 49, 'good': 50, 'more': 51, 'when': 52, 'very': 53, 'up': 54, 'no': 55, 'time': 56, 'she': 57, 'even': 58, 'my': 59, 'would': 60, 'which': 61, 'only': 62, 'story': 63, 'really': 64, 'see': 65, 'their': 66, 'had': 67, 'can': 68, 'were': 69, 'me': 70, 'well': 71, 'than': 72, 'we': 73, 'much': 74, 'been': 75, 'bad': 76, 'get': 77, 'will': 78, 'do': 79, 'also': 80, 'into': 81, 'people': 82, 'other': 83, 'first': 84, 'great': 85, 'because': 86, 'how': 87, 'him':

# Step 4: Padding

In [24]:
max_len = 6

X = pad_sequences(sequences, maxlen=max_len, padding='post')
y = np.array(labels)

# Step 5: Train-Test Split

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Step6: Embeddings

In [26]:
# Took 30 sec

from gensim.models import Word2Vec

# Tokenize text manually (Word2Vec needs list of words)
tokenized_texts = [text.lower().split() for text in texts]

# Train Word2Vec
w2v_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4
)

In [27]:
# Create Embedding Matrix
embedding_dim = 50
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in word_index.items():
    if i < vocab_size:
        if word in w2v_model.wv:
            embedding_matrix[i] = w2v_model.wv[word]


# Step 7: Build LSTM Model + Dropout

In [28]:
model = Sequential()

model.add(Embedding(input_dim=vocab_size,
                    output_dim=embedding_dim,
                    weights=[embedding_matrix],
                    input_length=max_len,
                    trainable=False))  # freeze embeddings

model.add(LSTM(32, return_sequences=False))

model.add(Dropout(0.5))  # prevent overfitting

model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 6, 50)             250000    
                                                                 
 lstm_1 (LSTM)               (None, 32)                10624     
                                                                 
 dropout_1 (Dropout)         (None, 32)                0         
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 260657 (1018.19 KB)
Trainable params: 10657 (41.63 KB)
Non-trainable params: 250000 (976.56 KB)
_________________________________________________________________


# Step 8: Train

In [29]:
model.fit(X_train, y_train, epochs=10, verbose=1)

Epoch 1/10
625/625 [==============================] - 5s 4ms/step - loss: 0.6400 - accuracy: 0.6188
Epoch 2/10
625/625 [==============================] - 2s 4ms/step - loss: 0.5913 - accuracy: 0.6738
Epoch 3/10
625/625 [==============================] - 2s 4ms/step - loss: 0.5689 - accuracy: 0.6898
Epoch 4/10
625/625 [==============================] - 2s 4ms/step - loss: 0.5550 - accuracy: 0.7004
Epoch 5/10
625/625 [==============================] - 2s 4ms/step - loss: 0.5437 - accuracy: 0.7077
Epoch 6/10
625/625 [==============================] - 2s 4ms/step - loss: 0.5368 - accuracy: 0.7118
Epoch 7/10
625/625 [==============================] - 2s 4ms/step - loss: 0.5303 - accuracy: 0.7146
Epoch 8/10
625/625 [==============================] - 2s 3ms/step - loss: 0.5191 - accuracy: 0.7261
Epoch 9/10
625/625 [==============================] - 2s 3ms/step - loss: 0.5116 - accuracy: 0.7337
Epoch 10/10
625/625 [==============================] - 2s 3ms/step - loss: 0.5065 - accuracy: 0.7373

# Step 9: Evaluate

In [30]:
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy:", acc)


157/157 [==============================] - 1s 3ms/step - loss: 0.5585 - accuracy: 0.6954
Accuracy: 0.6953999996185303


# Step 10: Test Custom Input

In [31]:
def predict_sentiment(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len, padding='post')
    pred = model.predict(padded)[0][0]
    
    return "Positive" if pred > 0.5 else "Negative"


print(predict_sentiment("I really love this movie"))
print(predict_sentiment("I hate this"))

1/1 [==============================] - 1s 600ms/step
Positive
1/1 [==============================] - 0s 28ms/step
Negative


# SWITCH TO REAL DATASET

In [18]:
# Here I used following code to create a CSV file that contains the necessary data

# from tensorflow.keras.datasets import imdb
# import pandas as pd

# vocab_size = 10000

# (X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

# word_index = imdb.get_word_index()
# reverse_word_index = {v: k for k, v in word_index.items()}


# def decode_review(sequence):
#     return " ".join([reverse_word_index.get(i - 3, "?") for i in sequence])


# train_texts = [decode_review(seq) for seq in X_train]
# test_texts = [decode_review(seq) for seq in X_test]



# df_train = pd.DataFrame({
#     "review": train_texts,
#     "sentiment": y_train
# })

# df_test = pd.DataFrame({
#     "review": test_texts,
#     "sentiment": y_test
# })

# df_train.to_csv("data_imdb_train.csv", index=False)
# df_test.to_csv("data_imdb_test.csv", index=False)